# parserInvestfunds — выгрузка ПИФов с pro.investfunds.ru **без фильтров**

Что поменялось против прежней редакции:

| было | стало |
|---|---|
| фильтры: тип (ОПИФ/БПИФ/ИПИФ) × категория × специализация, 6 книг | фильтров нет, отмечено всё → **один** набор файлов |
| периоды: полугодия, список дат руками | **кварталы** назад от `сегодня - DAYS_BACK` (X в хардкоде) |
| сшивка через Excel/xlwings + `strip_inner_headers` | скачанные файлы читаются по очереди в **один `DataFrame df`** |
| результат: `akcii_investfundsОПИФЫ.xlsx` и ещё 5 книг | результат: `df` в памяти (сохранение — последняя ячейка, по желанию) |

Excel на машине больше не нужен — вся сборка в pandas. Файлы падают в папку
рядом с кодом (`./investfunds_xlsx`), а не туда, где лежит venv.

**Плата за отказ от фильтров:** в выгрузке сайта нет колонок «тип фонда» и
«категория» — раньше их заменяло имя книги (`akcii_…ОПИФЫ`). В общем `df`
останутся только колонки сайта (Дата / Фонд / УК / Пай / СЧА); тип и категорию,
если понадобятся, придётся подтягивать отдельным справочником.

**Кодировка.** Кнопка CSV отдаёт файл в cp1251 (ANSI Windows), и `pd.read_csv`
с кодировкой по умолчанию падает на первом же байте слова «Дата»
(`'utf-8' codec can't decode byte 0xc4 in position 0`). Поэтому в ячейке 4
байты декодируются вручную (`decode_csv`: utf-8 → cp1251 → cp1251 с заменой),
а если сайт отдал уже испорченный текст («Äàòà» вместо «Дата»), его
переигрывает `fix_all_encoding`. Важно: эта функция срабатывает только на
кракозябрах (`is_mojibake`) — на правильной кириллице второй проход превратил
бы «Дата» в «????».

Порядок прогона: ячейка 1 (настройки) → 2 (утилиты) → 3 (скачивание, Selenium +
msedgedriver.exe) → 4 (сборка `df`) → 5 (сводка). Ячейку 4 можно крутить
отдельно, не перекачивая файлы. Ячейка 4b (одиночный файл → xlsx) нужна редко
и требует, чтобы ячейка 4 уже была прогнана.

In [50]:
# =============================================================================
# 1. Настройки и периоды
# =============================================================================
import datetime
import os
import re
import time
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------ хардкод ---
DAYS_BACK = 2        # X: правый край самого свежего периода = сегодня - X дней
N_QUARTERS = 22      # сколько кварталов тянуть назад (22 ~ с 3 кв. 2021)

LOGIN = "baxa"
PASSWORD = "mozart"

DOWNLOAD_TIMEOUT = 600   # сек на одну выгрузку: без фильтров файл крупный
SETTLE_AFTER_DATES = 2   # сек на пересборку таблицы под новый диапазон
RETRIES = 1              # повторов на период, если выгрузка не доехала
SKIP_EXISTING = True     # уже скачанный период повторно не тянуть
KEEP_SOURCE_COL = False  # оставить в df служебную колонку с именем файла

# --------------------------------------------------------------------- пути ---
# Папка с кодом, а не с venv. В Jupyter у ячейки нет __file__, поэтому берём cwd
# ядра: для ноутбука это его собственная папка. В .py-скрипте замените строку на
# BASE_DIR = Path(__file__).resolve().parent
BASE_DIR = Path.cwd()
DOWNLOAD_DIR = BASE_DIR / "investfunds_xlsx"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# msedgedriver.exe: сначала рядом с кодом, потом по известным местам.
# Selenium тут работает только с этим драйвером, поэтому путь ищем, а не угадываем.
DRIVER_CANDIDATES = (
    BASE_DIR / "msedgedriver.exe",
    Path(r"C:\Users\tmp_SKosyrkov\Downloads\py_ex\general_0\msedgedriver.exe"),
    Path.home() / "Downloads" / "msedgedriver.exe",
)


def resolve_driver(candidates=DRIVER_CANDIDATES) -> Path:
    """Первый существующий путь к msedgedriver.exe или понятная ошибка."""
    for p in candidates:
        if Path(p).is_file():
            return Path(p)
    raise FileNotFoundError(
        "msedgedriver.exe не найден. Положите его рядом с кодом "
        f"({BASE_DIR}) или допишите путь в DRIVER_CANDIDATES. Искал:\n  "
        + "\n  ".join(str(p) for p in candidates)
    )


# ------------------------------------------------------- кварталы от «сегодня»
SITE_FMT = "%d.%m.%Y"   # формат календаря на сайте


def _q_start(d: datetime.date) -> datetime.date:
    """Первое число квартала, в который попала дата."""
    return d.replace(month=3 * ((d.month - 1) // 3) + 1, day=1)


def quarter_periods(anchor: datetime.date, n: int):
    """
    n кварталов назад от anchor: [(начало, конец), ...], от свежего к старому.

    Самый свежий период обрезан по anchor — квартал ещё не закончился. Остальные
    полные: шаг «первое число квартала минус день» сам попадает на конец
    предыдущего квартала, так что длину квартала считать не нужно.
    """
    out, cur = [], anchor
    for _ in range(n):
        start = _q_start(cur)
        out.append((start, cur))
        cur = start - datetime.timedelta(days=1)
    return out


ANCHOR_DATE = datetime.date.today() - datetime.timedelta(days=DAYS_BACK)
PERIODS = quarter_periods(ANCHOR_DATE, N_QUARTERS)

print(f"сегодня {datetime.date.today():{SITE_FMT}}, якорь (-{DAYS_BACK} дн.) "
      f"{ANCHOR_DATE:{SITE_FMT}}, периодов {len(PERIODS)}")
print(f"файлы кладём в {DOWNLOAD_DIR}")
for s, e in PERIODS:
    print(f"  {s:{SITE_FMT}} - {e:{SITE_FMT}}")

сегодня 23.10.2026, якорь (-2 дн.) 21.10.2026, периодов 22
файлы кладём в c:\Users\tmp_SKosyrkov\Downloads\py_ex\peers_vs_funds_2\investfunds_xlsx
  01.10.2026 - 21.10.2026
  01.07.2026 - 30.09.2026
  01.04.2026 - 30.06.2026
  01.01.2026 - 31.03.2026
  01.10.2025 - 31.12.2025
  01.07.2025 - 30.09.2025
  01.04.2025 - 30.06.2025
  01.01.2025 - 31.03.2025
  01.10.2024 - 31.12.2024
  01.07.2024 - 30.09.2024
  01.04.2024 - 30.06.2024
  01.01.2024 - 31.03.2024
  01.10.2023 - 31.12.2023
  01.07.2023 - 30.09.2023
  01.04.2023 - 30.06.2023
  01.01.2023 - 31.03.2023
  01.10.2022 - 31.12.2022
  01.07.2022 - 30.09.2022
  01.04.2022 - 30.06.2022
  01.01.2022 - 31.03.2022
  01.10.2021 - 31.12.2021
  01.07.2021 - 30.09.2021


In [51]:
# =============================================================================
# 2. Утилиты файловой системы: ждём докачку, раскладываем файлы по периодам
# =============================================================================
PARTIAL_SUFFIXES = (".crdownload", ".tmp", ".partial")


def dir_snapshot(folder) -> set:
    return set(os.listdir(folder))


def wait_for_new_download(folder, before: set, timeout: int = DOWNLOAD_TIMEOUT) -> Path:
    """Ждёт появления нового полностью догруженного файла и возвращает его путь."""
    deadline = time.time() + timeout
    last_size = -1
    while time.time() < deadline:
        fresh = [f for f in os.listdir(folder)
                 if f not in before and not f.endswith(PARTIAL_SUFFIXES)]
        if fresh:
            path = max((Path(folder) / f for f in fresh),
                       key=lambda p: p.stat().st_mtime)
            size = path.stat().st_size
            if size > 0 and size == last_size:   # два замера подряд совпали
                return path
            last_size = size
        time.sleep(0.7)
    raise TimeoutError(f"Новый файл не появился в {folder} за {timeout} с")


def period_path(start, fin, suffix=".xlsx") -> Path:
    """
    Имя под период: investfunds_20260701_20260930.xlsx.

    Детерминированное имя вместо «Выгрузка (7).xls» даёт две вещи: сортировка
    по имени = сортировка по времени, и повторный прогон того же периода
    перезатирает файл, а не плодит копии.
    """
    return DOWNLOAD_DIR / f"investfunds_{start:%Y%m%d}_{fin:%Y%m%d}{suffix}"


def existing_period_file(start, fin):
    """Уже скачанный файл периода (любое расширение) или None."""
    hits = sorted(DOWNLOAD_DIR.glob(f"investfunds_{start:%Y%m%d}_{fin:%Y%m%d}.*"))
    return hits[0] if hits else None

def clear_filters(driver, wait, container_ids=("cat_check_list", "spec_check_list"),
                  settle=0.4, fast=True, attempts=2, strict=True):
    """
    Снимает все отмеченные чекбоксы в указанных контейнерах (на свежей странице
    отмечено всё). Один JS-проход на контейнер + добивание WebDriver-кликами.

    container_ids : строка или кортеж; порядок важен — см. замечание ниже
    fast          : снимать одним execute_script без round-trip'ов на каждый чекбокс
    attempts      : сколько раз повторить проход, если после перерисовки что-то вернулось
    strict        : RuntimeError, если снять всё так и не удалось

    Возвращает {container_id: сколько чекбоксов осталось отмеченными} (в норме нули).
    """
    if isinstance(container_ids, str):
        container_ids = (container_ids,)

    residual = {}
    for cid in container_ids:
        try:
            wait.until(lambda d, c=cid: d.find_elements(
                By.CSS_SELECTOR, f"#{c} input[type=checkbox]"))
        except TimeoutError:          # контейнера нет/пуст — например, spec после сброса категорий
            residual[cid] = 0
            continue

        for _ in range(attempts):
            if fast:
                driver.execute_script(
                    "document.querySelectorAll(arguments[0] + ' input[type=checkbox]')"
                    ".forEach(function (b) { if (b.checked) b.click(); });",
                    "#" + cid,
                )

In [52]:
# =============================================================================
# 3. Скачивание: Selenium + msedgedriver.exe, фильтры НЕ применяются
# =============================================================================
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)

# type_check_list  - тип фонда (ОПИФ/БПИФ/ИПИФ/ЗПИФ)
# cat_check_list   - категория (акции, облигации, ...)
# spec_check_list  - специализация внутри категории
FILTER_CONTAINERS = ("type_check_list", "cat_check_list", "spec_check_list")


def filter_state(driver, container_id):
    """(отмечено, всего) по контейнеру — чтобы глазами видеть, что фильтра нет."""
    boxes = driver.find_elements(
        By.CSS_SELECTOR, f"#{container_id} input[type=checkbox]")
    on = 0
    for b in boxes:
        try:
            on += bool(b.is_selected())
        except StaleElementReferenceException:
            pass
    return on, len(boxes)


def ensure_all_checked(driver, containers=FILTER_CONTAINERS, settle=0.6, attempts=2):
    """
    «Не фильтровать» = отмечено всё. На свежей странице так и есть, но профиль
    Edge (--user-data-dir) умеет помнить прошлый выбор, поэтому недостающие галки
    доставляем сами — иначе молча получим урезанную выгрузку.

    Порядок контейнеров важен: тип -> категории -> специализации, нижние списки
    перерисовываются обработчиком верхних. Клик делаем через JS .click(), а не
    присваиванием b.checked = true: присваивание не шлёт событий, и inline
    onclick=setspecs()/setuk() не выполнится.
    """
    report = {}
    for cid in containers:
        for _ in range(attempts):
            on, total = filter_state(driver, cid)
            if total and on == total:
                break
            driver.execute_script(
                "document.querySelectorAll(arguments[0] + ' input[type=checkbox]')"
                ".forEach(function (b) { if (!b.checked) b.click(); });",
                "#" + cid,
            )
            time.sleep(settle)
        report[cid] = filter_state(driver, cid)   # в норме (N, N) по каждому
    return report


DRIVER_PATH = resolve_driver()
print("драйвер:", DRIVER_PATH)

edge_options = webdriver.EdgeOptions()
# edge_options.add_argument("--headless=new")
edge_options.add_argument("--window-size=1920,1080")
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--use-system-proxy-resolver")
user_data_dir = os.path.expanduser("~") + r"\AppData\Local\Microsoft\User Data"
edge_options.add_argument(f"--user-data-dir={user_data_dir}")
edge_options.add_argument("--profile-directory=Default")
edge_options.add_experimental_option(
    "prefs",
    {
        # качаем рядом с кодом, а не в папку venv и не в системные «Загрузки»
        "download.default_directory": str(DOWNLOAD_DIR),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    },
)

driver = webdriver.Edge(
    options=edge_options,
    service=Service(executable_path=str(DRIVER_PATH)),
)
wait = WebDriverWait(driver, 20)

saved, failed = [], []
try:
    driver.get("https://pro.investfunds.ru/")
    driver.get("https://pro.investfunds.ru/quotes/")

    try:
        ln = driver.find_element(By.ID, "loginName")
        ln.click()
        ln.send_keys(LOGIN)
        pw = driver.find_element(By.ID, "loginPass")
        pw.click()
        pw.send_keys(PASSWORD)
        driver.find_element(
            By.CSS_SELECTOR, "input.header-user-login-submit[type='submit']"
        ).click()
        time.sleep(2)
    except NoSuchElementException:
        # профиль Edge помнит сессию — формы логина на странице просто нет
        print("форма логина не найдена, похоже уже залогинены")

    wait.until(EC.presence_of_element_located((By.ID, "csv")))
    print("фильтры (отмечено/всего):", ensure_all_checked(driver))
    
    clear_filters(driver, wait, container_ids=['type_check_list'])
    wait.until(EC.element_to_be_clickable((By.ID, "t1"))).click()

    for i, (start, fin) in enumerate(PERIODS, 1):
        tag = f"[{i}/{len(PERIODS)}] {start:{SITE_FMT}}-{fin:{SITE_FMT}}"

        have = existing_period_file(start, fin)
        if SKIP_EXISTING and have:
            print(f"{tag}: уже лежит, пропуск ({have.name})")
            saved.append(have)
            continue

        for attempt in range(RETRIES + 1):
            try:
                st_dat = driver.find_element(By.ID, "calendar-1")
                fin_dat = driver.find_element(By.ID, "calendar-2")

                st_dat.send_keys(Keys.CONTROL + "a")
                st_dat.send_keys(Keys.BACKSPACE)
                st_dat.send_keys(f"{start:{SITE_FMT}}")

                fin_dat.send_keys(Keys.CONTROL + "a")
                fin_dat.send_keys(Keys.BACKSPACE)
                fin_dat.send_keys(f"{fin:{SITE_FMT}}")
                fin_dat.send_keys(Keys.ESCAPE)   # закрыть всплывающий календарь

                time.sleep(SETTLE_AFTER_DATES)   # дать таблице пересобраться

                before = dir_snapshot(DOWNLOAD_DIR)
                driver.find_element(By.ID, "csv").click()
                got = wait_for_new_download(DOWNLOAD_DIR, before)

                dst = period_path(start, fin, got.suffix.lower() or ".xlsx")
                os.replace(got, dst)
                saved.append(dst)
                print(f"{tag}: {dst.name}, {dst.stat().st_size / 1024:.0f} КБ")
                break
            except (TimeoutError, TimeoutException) as err:
                # один период не доехал — остальные всё равно забираем
                print(f"{tag}: попытка {attempt + 1} мимо — {err}")
                if attempt == RETRIES:
                    failed.append((f"{start:{SITE_FMT}}", f"{fin:{SITE_FMT}}", repr(err)))
finally:
    driver.quit()

print(f"\nскачано {len(saved)}, провалов {len(failed)}")
for f in failed:
    print("  провал:", f)

драйвер: C:\Users\tmp_SKosyrkov\Downloads\py_ex\general_0\msedgedriver.exe
фильтры (отмечено/всего): {'type_check_list': (4, 4), 'cat_check_list': (18, 18), 'spec_check_list': (88, 88)}
[1/22] 01.10.2026-21.10.2026: investfunds_20261001_20261021.csv, 277 КБ
[2/22] 01.07.2026-30.09.2026: investfunds_20260701_20260930.csv, 1176 КБ
[3/22] 01.04.2026-30.06.2026: investfunds_20260401_20260630.csv, 1081 КБ
[4/22] 01.01.2026-31.03.2026: investfunds_20260101_20260331.csv, 938 КБ
[5/22] 01.10.2025-31.12.2025: investfunds_20251001_20251231.csv, 1062 КБ
[6/22] 01.07.2025-30.09.2025: investfunds_20250701_20250930.csv, 1050 КБ
[7/22] 01.04.2025-30.06.2025: investfunds_20250401_20250630.csv, 929 КБ
[8/22] 01.01.2025-31.03.2025: investfunds_20250101_20250331.csv, 913 КБ
[9/22] 01.10.2024-31.12.2024: investfunds_20241001_20241231.csv, 1034 КБ
[10/22] 01.07.2024-30.09.2024: investfunds_20240701_20240930.csv, 1037 КБ
[11/22] 01.04.2024-30.06.2024: investfunds_20240401_20240630.csv, 944 КБ
[12/22] 01.01.

In [53]:
# =============================================================================
# 4b. Старый путь «один файл -> одна книга» (необязательный)
#     Нужен, только если файл периода надо отдать в Excel отдельно. Прогонять
#     ПОСЛЕ ячейки 4: берёт её read_sheet / extract_block / typed, а значит и
#     ремонт кодировки (fix_all_encoding) — своей копии больше не держит.
#
#     Прежняя версия чинила кодировку безусловным df.map(fix_all_encoding).
#     Так делать нельзя: на правильной кириллице второй проход даёт '????'.
# =============================================================================
CLEAN_OUT_DIR = Path(r"C:/Users/tmp_SKosyrkov/Documents/files_for_peers_v_funds/test")


def clean_uploaded_df(path_to_that_csv, start_date, finish_date):
    """Файл периода -> xlsx без мусора."""
    part, _, _ = extract_block(read_sheet(path_to_that_csv), str(path_to_that_csv))
    part, date_col, _ = typed(part)

    part = part[part[date_col].notna()]                     # подвальные примечания
    part = part[part["УК"] != "ТКБ Инвестмент Партнерс"]
    part = part[part["СЧА"].notna()]                        # было "Не раскрывается"
    part = part.drop_duplicates()                           # повтор блока в хвосте файла

    CLEAN_OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = CLEAN_OUT_DIR / (f"funds_{start_date.replace('.', '')}"
                           f"_{finish_date.replace('.', '')}.xlsx")
    part.to_excel(out, index=False)
    return str(out)

In [54]:
# =============================================================================
# 4. Сборка одного DataFrame из скачанных файлов
#    (замена прежней связки xlwings + strip_inner_headers)
# =============================================================================
import csv
import io

HEADER_VALUES = ("Дата", "Фонд", "УК", "Пай", "СЧА")
_WANT = {h.lower() for h in HEADER_VALUES}
_NAME_RE = re.compile(r"investfunds_(\d{8})_(\d{8})")


def _norm(v) -> str:
    """Ячейка без nbsp, двойных пробелов и регистра — для сравнения со шапкой."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return re.sub(r"\s+", " ", str(v).replace("\xa0", " ")).strip().lower()


# ----------------------------------------------------------- кодировка CSV ----
# Кнопка CSV на сайте отдаёт файл в cp1251 (ANSI Windows), поэтому pandas с
# кодировкой по умолчанию падает на первом же байте слова «Дата» (0xC4):
# UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc4 in position 0.
# Лечим в два слоя: правильно декодируем байты файла, а если сайт отдал уже
# испорченный текст (cp1251, прочитанный как latin1: «Äàòà» вместо «Дата»),
# переигрываем декодирование через fix_all_encoding.
CSV_SUFFIXES = (".csv", ".txt", "")
CSV_ENCODINGS = ("utf-8-sig", "cp1251")   # порядок важен: utf-8 строгая и отвалится сама
CSV_SEPS = (";", "\t", ",", "|")

_CYRILLIC = re.compile(r"[Ѐ-ӿ]")
# Кириллица cp1251 (байты A8, B8, C0-FF), прочитанная как latin1/cp1252, даёт
# символы из этого диапазона. Требуем серию из 2+ подряд: одинокое «é» в
# латинском названии фонда — не кракозябра.
_MOJIBAKE_RUN = re.compile(r"[¨¸À-ÿ]{2,}")


def is_mojibake(v) -> bool:
    """Строка похожа на cp1251, прочитанный чужой кодировкой."""
    if not isinstance(v, str):
        return False
    bad = sum(len(run) for run in _MOJIBAKE_RUN.findall(v))
    return bad > 0 and bad > 10 * len(_CYRILLIC.findall(v))


def fix_all_encoding(text):
    """
    Кракозябры -> читаемый текст: байты возвращаем как latin1 и перечитываем
    как cp1251.

    Проверка is_mojibake обязательна, а не для красоты: на правильной строке
    второй проход всё ломает — cp1252 не умеет кириллицу, 'Дата' превратится
    в '????'. Нестроки и уже читаемый текст отдаём как есть.
    """
    if not isinstance(text, str) or not is_mojibake(text):
        return text
    try:
        return text.encode("latin1").decode("cp1251")
    except UnicodeEncodeError:
        # упало на спецсимволах (тире, кавычки) — подменяем их через cp1252
        return text.encode("cp1252", errors="replace").decode("cp1251", errors="replace")
    except Exception:
        return text


def fix_frame_encoding(frame: pd.DataFrame, name: str = "") -> pd.DataFrame:
    """
    Тот же ремонт, но для готового кадра (xlsx/HTML, где байтов уже не видно).

    Решение принимаем один раз на файл по образцу из первых строк, а не на
    каждую ячейку: на 50 тыс. строк это заметная разница, а кодировка внутри
    одного файла не меняется.
    """
    sample = [v for v in frame.head(50).to_numpy().ravel() if isinstance(v, str)]
    if not any(is_mojibake(v) for v in sample):
        return frame
    print(f"{name}: текст в кракозябрах — чиню fix_all_encoding")
    return frame.apply(lambda col: col.map(fix_all_encoding))


def decode_csv(raw: bytes, name: str = "") -> str:
    """Байты файла -> текст: сначала честные кодировки, потом ремонт."""
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):   # редкий, но встречающийся экспорт
        return raw.decode("utf-16")
    for enc in CSV_ENCODINGS:
        try:
            text = raw.decode(enc)
        except UnicodeDecodeError:
            continue
        if is_mojibake(text):
            print(f"{name}: {enc} дал кракозябры — чиню fix_all_encoding")
            text = fix_all_encoding(text)
        return text
    # ни одна не подошла: cp1251 с заменой битых байт, чтобы не потерять файл
    print(f"{name}: ни одна из {CSV_ENCODINGS} не подошла, читаю cp1251 с заменой")
    return raw.decode("cp1251", errors="replace")


def sniff_sep(text: str) -> str:
    """
    Разделитель по первым строкам. Считать голые запятые нельзя: в числах
    '1 234,56' их полно. Поэтому берём тот символ, который даёт самую широкую
    и при этом самую повторяемую ширину строки.
    """
    sample = text.splitlines()[:50]
    best, best_score = CSV_SEPS[0], 0
    for sep in CSV_SEPS:
        widths = [len(row) for row in csv.reader(sample, delimiter=sep)]
        if not widths:
            continue
        top = max(set(widths), key=widths.count)
        score = (top - 1) * widths.count(top)
        if top > 1 and score > best_score:
            best, best_score = sep, score
    return best


def csv_frame(text: str) -> pd.DataFrame:
    """
    Текст -> кадр без шапки. Читаем модулем csv, а не токенайзером pandas:
    в выгрузке строки рваные (заголовок периода сверху — одно поле на строку),
    а pandas на такой ширине падает «Expected 1 fields, saw 5». Короткие
    строки добиваем пустыми ячейками — их потом снимет extract_block.
    """
    rows = list(csv.reader(io.StringIO(text), delimiter=sniff_sep(text)))
    if not rows:
        return pd.DataFrame()
    width = max(len(r) for r in rows)
    return pd.DataFrame([r + [""] * (width - len(r)) for r in rows], dtype=object)


def read_sheet(path) -> pd.DataFrame:
    """
    Файл периода как есть, без шапки: CSV в cp1251, xlsx/xls или HTML-таблица
    под расширением .xls (кнопка XLS на сайте иногда отдаёт именно её, и на
    ней read_excel падает — выручает read_html).
    """
    path = Path(path)
    problems = []
    if path.suffix.lower() in CSV_SUFFIXES:
        try:
            text = decode_csv(path.read_bytes(), path.name)
            if text.lstrip()[:1] == "<":        # вместо CSV приехала HTML-страница
                raise ValueError("файл начинается с '<' — это HTML, не CSV")
            return fix_frame_encoding(csv_frame(text), path.name)
        except Exception as err_csv:
            problems.append(f"как CSV ({err_csv})")
    else:
        try:
            return fix_frame_encoding(
                pd.read_excel(path, header=None, dtype=object), path.name)
        except Exception as err_excel:
            problems.append(f"как Excel ({err_excel})")

    # Последний шанс: HTML-таблица под чужим расширением. Кодировку и здесь
    # берём на себя — read_html на cp1251-байтах с чужой кодировкой молча
    # подставит U+FFFD, а такой текст уже ничем не починить.
    try:
        # thousands=None: иначе read_html прочитает '721,39' как 72139
        tables = pd.read_html(
            io.StringIO(decode_csv(path.read_bytes(), path.name)), thousands=None)
    except Exception as err_html:
        tables = None
        problems.append(f"как HTML ({err_html})")
    if not tables:
        raise RuntimeError(f"{path.name}: не читается " + ", ни ".join(problems))
    t = max(tables, key=len)
    if not isinstance(t.columns, pd.RangeIndex):   # шапку забрал <th> — вернём строкой
        head = pd.DataFrame([list(t.columns)])
        t = pd.concat([head, t.set_axis(range(t.shape[1]), axis=1)], ignore_index=True)
    return fix_frame_encoding(t.set_axis(range(t.shape[1]), axis=1).astype(object), path.name)


def extract_block(df: pd.DataFrame, name: str):
    """
    То же, что прежние ANCHOR='C4' + Ctrl+A + strip_inner_headers, но в pandas:
    ищем строку-шапку, берём только её непустые столбцы (так отваливаются пустые
    A/B и заголовок выгрузки сверху), ниже выкидываем повторы шапки и пустые строки.

    Возвращает (кадр, убрано_шапок, убрано_пустых).
    """
    norm = df.apply(lambda col: col.map(_norm))

    hdr = None
    for i in range(len(norm)):
        # >=3 совпадений, а не все 5: страховка, если сайт переименует столбец
        if len(set(norm.iloc[i]) & _WANT) >= 3:
            hdr = i
            break
    if hdr is None:
        raise ValueError(
            f"{name}: не нашёл строку-шапку {HEADER_VALUES}. Если в первых "
            f"строках видно 'Äàòà' — кракозябра проехала мимо fix_all_encoding")
    cols = [j for j in range(df.shape[1]) if norm.iat[hdr, j]]
    head = [re.sub(r"\s+", " ", str(df.iat[hdr, j]).replace("\xa0", " ")).strip()
            for j in cols]

    body = df.iloc[hdr + 1:, cols]
    if body.empty:
        return pd.DataFrame(columns=head), 0, 0

    bn = norm.iloc[hdr + 1:, cols]
    key = bn.apply(lambda r: "|".join(x for x in r if x), axis=1)  # ключ строки без пустых
    hdr_key = "|".join(x for x in norm.iloc[hdr, cols] if x)

    is_hdr = key == hdr_key
    is_empty = key == ""
    out = body.loc[~(is_hdr | is_empty)].copy()
    out.columns = head
    return out.reset_index(drop=True), int(is_hdr.sum()), int(is_empty.sum())


def to_num(s: pd.Series) -> pd.Series:
    """'1 234,56' / 1234.56 -> float; прочерки и пустые -> NaN."""
    t = (s.astype(str)
          .str.replace("\xa0", "", regex=False)
          .str.replace(" ", "", regex=False)
          .str.replace(",", ".", regex=False))
    return pd.to_numeric(t, errors="coerce")


def to_date(s: pd.Series) -> pd.Series:
    """
    В столбце мешаются строки 'дд.мм.гггг' и настоящие datetime (их распознал
    openpyxl). Формат для строк задаём явно: pd.to_datetime('01.02.2026') без
    формата — это 2 января, а не 1 февраля.
    """
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    as_str = s.map(lambda v: isinstance(v, str))
    if as_str.any():
        out.loc[as_str] = pd.to_datetime(
            s[as_str].str.strip(), format="%d.%m.%Y", errors="coerce")
    if (~as_str).any():
        out.loc[~as_str] = pd.to_datetime(s[~as_str], errors="coerce")
    return out


def typed(part: pd.DataFrame):
    """Даты -> datetime, Пай/СЧА -> float, текст -> без nbsp и двойных пробелов."""
    date_col = next(c for c in part.columns if _norm(c) == "дата")
    num_cols = [c for c in part.columns if _norm(c) in {"пай", "сча"}]

    part[date_col] = to_date(part[date_col])
    for c in num_cols:
        part[c] = to_num(part[c])
    for c in part.columns:
        if c in (date_col, *num_cols) or c.startswith("_"):
            continue
        part[c] = part[c].map(
            lambda v: re.sub(r"\s+", " ", v.replace("\xa0", " ")).strip()
            if isinstance(v, str) else v)
    return part, date_col, num_cols


def period_from_name(name: str):
    """(начало, конец) из имени investfunds_20260701_20260930.xlsx или None."""
    m = _NAME_RE.match(name)
    return tuple(pd.Timestamp(g) for g in m.groups()) if m else None


# ---------------------------------------------------------------- сборка ------
files = sorted(DOWNLOAD_DIR.glob("investfunds_*"))
if not files:
    raise FileNotFoundError(
        f"в {DOWNLOAD_DIR} нет файлов investfunds_* — сначала прогоните ячейку скачивания")

frames, suspicious = [], []
for f in files:
    part, n_hdr, n_empty = extract_block(read_sheet(f), f.name)
    part, date_col, _ = typed(part)
    part["_файл"] = f.name          # след источника; снимается ниже по KEEP_SOURCE_COL

    span = "пусто"
    if not part.empty and part[date_col].notna().any():
        lo, hi = part[date_col].min(), part[date_col].max()
        span = f"{lo:{SITE_FMT}}-{hi:{SITE_FMT}}"
        want = period_from_name(f.name)
        # Если календарь на сайте не применился, файл приедет с данными чужого
        # периода. Дальше дедуп такие копии молча склеит, поэтому ловим здесь.
        if want and (lo < want[0] or hi > want[1]):
            suspicious.append(f.name)
            span += " (!) вне периода из имени"

    frames.append(part)
    print(f"{f.name}: {len(part)} строк, {span} "
          f"(шапок убрано {n_hdr}, пустых {n_empty})")

layouts = {tuple(p.columns) for p in frames}
if len(layouts) > 1:
    raise ValueError("файлы с разным набором столбцов: "
                     + "; ".join(str(sorted(l)) for l in layouts))

df = pd.concat(frames, ignore_index=True)

DATE_COL = next(c for c in df.columns if _norm(c) == "дата")
NUM_COLS = [c for c in df.columns if _norm(c) in {"пай", "сча"}]
TXT_COLS = [c for c in df.columns if c not in (DATE_COL, *NUM_COLS, "_файл")]

no_date = int(df[DATE_COL].isna().sum())      # подвальные примечания выгрузки
df = df.loc[df[DATE_COL].notna()]

was = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "_файл"])
dups = was - len(df)

df = df.sort_values([DATE_COL, *TXT_COLS], kind="stable").reset_index(drop=True)
if not KEEP_SOURCE_COL:
    df = df.drop(columns="_файл")

print(f"\nитог: {len(df)} строк, {len(df.columns)} столбцов; "
      f"без даты отброшено {no_date}, дублей снято {dups}")
if not df.empty:
    print(f"период {df[DATE_COL].min():{SITE_FMT}} - {df[DATE_COL].max():{SITE_FMT}}, "
          f"дат {df[DATE_COL].nunique()}, фондов {df[TXT_COLS[0]].nunique()}")
if suspicious:
    print("ВНИМАНИЕ: даты не применились в файлах — перекачайте их, "
          "иначе период потерян:\n  " + "\n  ".join(suspicious))
df.head()

investfunds_20210701_20210930.csv: 13090 строк, 01.07.2021-30.09.2021 (шапок убрано 0, пустых 61)
investfunds_20211001_20211231.csv: 12579 строк, 01.10.2021-30.12.2021 (шапок убрано 0, пустых 63)
investfunds_20220101_20220331.csv: 7620 строк, 09.01.2022-31.03.2022 (шапок убрано 0, пустых 64)
investfunds_20220401_20220630.csv: 9122 строк, 01.04.2022-30.06.2022 (шапок убрано 0, пустых 49)
investfunds_20220701_20220930.csv: 11189 строк, 01.07.2022-30.09.2022 (шапок убрано 0, пустых 51)
investfunds_20221001_20221231.csv: 11209 строк, 03.10.2022-30.12.2022 (шапок убрано 0, пустых 54)
investfunds_20230101_20230331.csv: 10489 строк, 09.01.2023-31.03.2023 (шапок убрано 0, пустых 57)
investfunds_20230401_20230630.csv: 11711 строк, 03.04.2023-30.06.2023 (шапок убрано 0, пустых 60)
investfunds_20230701_20230930.csv: 12799 строк, 03.07.2023-29.09.2023 (шапок убрано 0, пустых 59)
investfunds_20231001_20231231.csv: 13095 строк, 02.10.2023-29.12.2023 (шапок убрано 0, пустых 62)
investfunds_20240101_2

,Дата,Фонд,УК,Пай,СЧА
0,2021-07-01,А-Капитал Валютные облигации,А-Капитал,205.57,1.728619e+09
1,2021-07-01,А-Капитал Облигации,А-Капитал,342.50,1.356622e+09
2,2021-07-01,А-Капитал Сбалансированный,А-Капитал,2105.99,3.325382e+08
3,2021-07-01,АК БАРС - Акции,АК БАРС КАПИТАЛ,3856.64,4.649357e+06
4,2021-07-01,АК БАРС - Консервативный с выплатой дохода,АК БАРС КАПИТАЛ,724.27,1.412097e+08


In [55]:
df.to_excel('df_zero.xlsx')

In [56]:
df['Дата'].unique()

<DatetimeArray>
['2021-07-01 00:00:00', '2021-07-02 00:00:00', '2021-07-05 00:00:00',
 '2021-07-06 00:00:00', '2021-07-07 00:00:00', '2021-07-08 00:00:00',
 '2021-07-09 00:00:00', '2021-07-12 00:00:00', '2021-07-13 00:00:00',
 '2021-07-14 00:00:00',
 ...
 '2026-10-08 00:00:00', '2026-10-09 00:00:00', '2026-10-12 00:00:00',
 '2026-10-13 00:00:00', '2026-10-14 00:00:00', '2026-10-15 00:00:00',
 '2026-10-16 00:00:00', '2026-10-19 00:00:00', '2026-10-20 00:00:00',
 '2026-10-21 00:00:00']
Length: 1341, dtype: datetime64[ns]

In [57]:
duplicates = df[df.duplicated(subset=['Дата', 'Фонд'])]
duplicates

,Дата,Фонд,УК,Пай,СЧА


In [58]:
df_sorted = df.copy().sort_values(by=['Фонд', 'Дата'])
df['Фонд'].unique().tolist()

['А-Капитал Валютные облигации',
 'А-Капитал Облигации',
 'А-Капитал Сбалансированный',
 'АК БАРС - Акции',
 'АК БАРС - Консервативный с выплатой дохода',
 'АК БАРС – Цифровые лидеры',
 'Агидель - акции',
 'Активные инвестиции',
 'Акции роста',
 'Алгоритмический',
 'Альтернативный процент',
 'Альфа-Капитал Баланс',
 'Альфа-Капитал Ликвидные акции',
 'Альфа-Капитал Облигации Плюс',
 'Альфа-Капитал Ресурсы',
 'Алёнка - Капитал',
 'Апрель Капитал - Акции',
 'Апрель Капитал - Акции несырьевых компаний',
 'Апрель Капитал - Акции сырьевых компаний',
 'Апрель Капитал - Облигации плюс',
 'Апрель Капитал - Сбалансированный',
 'Арикапитал - Чистые деньги',
 'Арикапитал – Мировые рынки',
 'Арикапитал – Мультистратегия',
 'Арикапитал – Финансовая свобода',
 'Арсагера - акции 6.4',
 'Арсагера - фонд акций',
 'Арсагера - фонд облигаций КР 1.55',
 'Арсагера - фонд смешанных инвестиций',
 'Атлант',
 'Атон - Петр Столыпин',
 'Атон - Фонд Еврооблигаций',
 'Атон - Фонд высоких технологий',
 'Атон - Фонд 

In [59]:
df_wide = pd.DataFrame()
df_wide.index = sorted(df['Дата'].unique())
df_1 = df.pivot(index='Дата', columns='Фонд', values='СЧА').sort_index()

In [63]:
df_1

Фонд,А-Капитал Акции,А-Капитал Валютные облигации,А-Капитал Золото,А-Капитал Облигации,А-Капитал Облигации Рантье,А-Капитал Сбалансированный,ААА Акции российских компаний,ААА Сбалансированный,АК БАРС - Акции,АК БАРС - Консервативный с выплатой дохода,...,Фонд Консервативный,Фонд Первый,Фонд новые технологии,Фонд природных ресурсов,Харизматичные идеи,Энергетическая перспектива,Яндекс Пэй. Фонд акций и облигаций,Яндекс Пэй. Фонд денежного рынка. Рубли,Яндекс Пэй. Фонд золото,Яндекс Пэй. Фонд рублёвых облигаций
Дата,,,,,,,,,,,,,,,,,,,,,
2021-07-01,NaN,1.728619e+09,NaN,1.356622e+09,NaN,3.325382e+08,NaN,NaN,4649357.21,1.412097e+08,...,2.891493e+09,6.022878e+09,19981285.16,NaN,2.901607e+08,2.273397e+08,NaN,NaN,NaN,NaN
2021-07-02,NaN,1.748402e+09,NaN,1.357498e+09,NaN,3.332205e+08,NaN,NaN,4642022.72,1.410761e+08,...,2.881616e+09,6.013300e+09,19978846.28,NaN,2.888333e+08,2.271688e+08,NaN,NaN,NaN,NaN
2021-07-05,NaN,1.768828e+09,NaN,1.356888e+09,NaN,3.347590e+08,NaN,NaN,4690088.94,1.414988e+08,...,2.889879e+09,6.017575e+09,23978407.51,NaN,2.936815e+08,2.296457e+08,NaN,NaN,NaN,NaN
2021-07-06,NaN,1.761112e+09,NaN,1.356635e+09,NaN,3.363273e+08,NaN,NaN,4661408.54,1.430995e+08,...,2.886514e+09,6.009908e+09,19976968.83,NaN,2.931583e+08,2.270728e+08,NaN,NaN,NaN,NaN
2021-07-07,NaN,1.768509e+09,NaN,1.356689e+09,NaN,3.342604e+08,NaN,NaN,4671022.09,1.431438e+08,...,2.881086e+09,6.043081e+09,19975530.26,NaN,2.929747e+08,2.269513e+08,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-15,6.430090e+08,8.838026e+08,5.071071e+08,2.037402e+09,3.648253e+08,3.975350e+08,8640264.05,11607778.88,3907076.95,2.088376e+07,...,2.659874e+09,3.273138e+09,NaN,2.339740e+08,6.505446e+08,NaN,8.511444e+07,4.389122e+09,15000675.36,82553874.98
2026-10-16,6.532081e+08,8.785074e+08,5.004819e+08,2.028219e+09,3.649398e+08,4.025362e+08,8753231.23,11612276.25,3939437.77,2.087471e+07,...,2.660849e+09,3.311394e+09,NaN,2.379028e+08,6.644859e+08,NaN,8.478499e+07,4.153833e+09,14965435.07,62930747.43
2026-10-19,6.495239e+08,8.824099e+08,4.967964e+08,2.045425e+09,3.653587e+08,4.053409e+08,8789480.24,11623764.52,3948775.35,2.092742e+07,...,2.664588e+09,3.336981e+09,NaN,2.394657e+08,6.699289e+08,NaN,8.407759e+07,4.107494e+09,14894370.09,63072340.31


In [ ]:
df_1.index

In [ ]:
df_1[df_1['дата']]